## Creating the training dataset from the equilibria dataset

In [ ]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"]="-1" # disable GPU
import numpy as np
from time import time, sleep
from os.path import join, exists
from utils import *
from tqdm import tqdm
print("Preparing data...")

In [ ]:
# hyperparameters
RAW_DIR = 'raw_ds' if LOCAL else '/nfsd/automatica/grandinmat/raw_ds' # where the raw data is stored

# hyperparameters
N_SAMPLES = 1000 if LOCAL else 250_000 #1_000_000 #500_000 #850_000 # number of samples to use for training
SM = 1000 if LOCAL else 2500 #1400 # 20 # number of points per sample (SM = SAMPLE MULTIPLIER)
N_SEP_PTS = SM #int(0.5 * SM) # number of points to sample from close to the LCFS
LCFS_FX_SIGMA = 0.3 # 0.12 # standard deviation for the Fx-based sampling around the LCFS
TRAIN_EVAL_SPLIT = 0.8 # percentage of the dataset to use for training

MIN_NUMBER_NONZERO_IY = 240
LCFS_INNER_THRSH = 0.25 # threshold for the inner LCFS points

# function to use for sampling points
# SAMPLING_FUNC = sample_points_fx_importance_samp 
SAMPLING_FUNC = uniform_sampling_inside_fw 

print(f'Total samples: {N_SAMPLES:.0f}, train samples: {N_SAMPLES*TRAIN_EVAL_SPLIT:.0f}, eval samples: {N_SAMPLES*(1-TRAIN_EVAL_SPLIT):.0f}')

In [ ]:
# check RAW_DIR exists and is not empty
assert exists(RAW_DIR), f'RAW_DIR {RAW_DIR} does not exist'
assert len(os.listdir(RAW_DIR)) > 0, f'RAW_DIR {RAW_DIR} is empty'

In [ ]:
def plot_example(fx=None, iy=None, br=None, bz=None, rq=None, zq=None, bm=None, ff=None, ia=None, rr=RRD, zz=ZZD, title='Example'):
    plt.figure(figsize=(15, 8))
    rq,zq = filter_zeros(rq, zq)
    if fx is not None:
        plt.subplot(2, 4, 1)
        plt.scatter(rr, zz, c=fx, s=4), plt.title('FX')
        plt.plot(rq, zq, 'gray', lw=2)
        plot_vessel(), plt.axis('equal'), plt.axis('off'), plt.colorbar()
    if iy is not None:
        plt.subplot(2, 4, 2)
        plt.scatter(rr, zz, c=iy, s=4), plt.title('IY')
        plt.plot(rq, zq, 'gray', lw=2)
        plot_vessel(), plt.axis('equal'), plt.axis('off'), plt.colorbar()
    if br is not None:
        plt.subplot(2, 4, 3)
        plt.scatter(rr, zz, c=br, s=4), plt.title('BR')
        plt.plot(rq, zq, 'gray', lw=2)
        plot_vessel(), plt.axis('equal'), plt.axis('off'), plt.colorbar()
    if bz is not None:
        plt.subplot(2, 4, 4)
        plt.scatter(rr, zz, c=bz, s=4), plt.title('BZ')
        plt.plot(rq, zq, 'gray', lw=2)
        plot_vessel(), plt.axis('equal'), plt.axis('off'), plt.colorbar()
    if bm is not None:
        plt.subplot(2, 4, 5)
        plt.bar(np.arange(len(bm)), bm), plt.title('BM')
    if ff is not None:
        plt.subplot(2, 4, 6)
        plt.bar(np.arange(len(ff)), ff), plt.title('FF')
    if ia is not None:
        plt.subplot(2, 4, 7)
        plt.bar(np.arange(len(ia)), ia), plt.title('IA')
    plt.suptitle(f'{title}')
    plt.tight_layout()
    plt.show() if LOCAL else plt.savefig(f'{DS_DIR}/imgs/{title}.png')
    plt.close()

In [ ]:
# load the data
print("Loading data...")

# list all the files inside RAW_DIR/ds
files = sorted([f for f in os.listdir(RAW_DIR) if f.endswith('.mat')])
print(f'Found {len(files)} files.')
vals = {n:[] for n in DS_NAMES}  # dictionary to hold the numpy arrays
files_iter = tqdm(files, desc="Loading files", unit="file") if LOCAL else files
tot_samples = 0
for f in files_iter:
    try:
    # if True:
        d = loadmat(join(RAW_DIR, f))
        for n in DS_NAMES: assert n in list(d.keys()) or n in [BR, BZ], f'Key {n} not found in {f}' 
        t = d['t'].flatten()  # time
        nt = t.shape[0]  # number of time points
        tot_samples += nt
        to_keep_mask = np.ones(nt, dtype=bool)  # mask to keep all time points

        # filter out Iy maps too concentrated
        Iy_mask = np.abs(d[IY]) > 1e-2
        Iy_count = np.sum(Iy_mask, axis=(0, 1))  # count the number of non-zero elements in each Iy map
        to_keep_mask &= Iy_count > MIN_NUMBER_NONZERO_IY

        # filter out maps without lcfs
        rq, zq = d[RQ], d[ZQ]
        lcfs_filter = np.sum(rq**2 + zq**2, axis=0) > 1e-2 # lcfs points are non-zero
        to_keep_mask &= lcfs_filter

        # filter out maps with too low Ip
        ip = np.abs(d[IP].flatten())
        ip_filter = ip > 1e3
        # if np.sum(~ip_filter) > 0: print(f'Filtered out {np.sum(~ip_filter)}/{nt} samples with low Ip in {f}')
        to_keep_mask &= ip_filter

        if np.sum(~to_keep_mask) > 0: print(f'Filtered out {np.sum(~to_keep_mask)}/{nt} samples in {f}')

        # # plot filtered out examples
        # if LOCAL and np.sum(~to_keep_mask) > 0:
        #     pidx = np.where(~to_keep_mask)[0][0]
        #     tIy = np.zeros_like(d[FX][..., pidx])
        #     tIy[1:-1, 1:-1] = d[IY][..., pidx]  # fill the inner part with IY values
        #     plot_example(
        #         fx=d[FX][..., pidx], 
        #         iy=tIy, 
        #         rq=d[RQ][..., pidx], 
        #         zq=d[ZQ][..., pidx],
        #         title=f'Filtered out example {f} at index {pidx}'
        #     )
        # # plot only ip
        # if LOCAL and np.sum(~to_keep_mask) > 0:
        #     pidx = np.where(~to_keep_mask)[0]
        #     plt.figure(figsize=(10, 1.5))
        #     plt.scatter(t, ip, color='blue', s=1)
        #     plt.scatter(t[pidx], ip[pidx], color='red')

        #     plt.show()
        #     n_tot_plot += 1
        #     if n_tot_plot >= 50: break


        for n in DS_NAMES:
            if n in [BR, BZ]: continue  # skip BR and BZ for now, they will be calculated later
            v = d[n] # get the variable
            v = v.reshape((*DS_SIZES[n], nt)) # try to reshape it to the expected shape
            assert v.shape == (*DS_SIZES[n], nt), f'Variable {n} shape mismatch: {v.shape} != {((*DS_SIZES[n], nt))}'
            assert not np.isnan(v).any(), f'Variable {n} contains NaN values in {f}'
            assert np.isfinite(v).all(), f'Variable {n} contains infinite values in {f}'
            # vals[n].append(v)  # append to the list
            vals[n].append(v[..., to_keep_mask])  # append only the values that pass the mask
    except Exception as e:
        print(f'Error loading {f}: {e}')
        continue


print(f'Loaded {len(vals[FX])} files.')
assert len(vals[FX]) > 0, f'No samples: {len(vals[FX])}'

# convert lists to numpy arrays
print('---------------------------------------------------')
for n in DS_NAMES:
    if n in [BR, BZ]: continue  # skip BR and BZ for now, they will be calculated later
    vals[n] = np.concatenate(vals[n], axis=-1).astype(DTYPE)  # concatenate along the first dimension and convert to DTYPE
    sh, ndim = vals[n].shape, vals[n].ndim
    vals[n] = vals[n].transpose(ndim-1, *range(ndim-1))  # move the last dimension to the first
    # print(f'{n} -> shape: {vals[n].shape}, dtype: {vals[n].dtype}')  # print the shape and dtype of the variable
print('---------------------------------------------------')

# calculate BR and BZ from Fx
print('Calculating BR and BZ...')  
# vals[BR], vals[BZ] = meqBrBz(vals[FX])
vals[BR], vals[BZ] = np.zeros_like(vals[FX]), np.zeros_like(vals[FX])  # initialize BR and BZ with zeros

for n in DS_NAMES: print(f'{n} -> shape: {vals[n].shape}, dtype: {vals[n].dtype}')  # print the shape and dtype of the variable

for n in INPUT_NAMES: assert vals[n].ndim == 2, f'Variable {n} is not 2D: {vals[n].ndim}'

# assign values to variables
X = np.concatenate([vals[n] for n in INPUT_NAMES], axis=1)  # concatenate along the first dimension
print(f'-----------------------------------------\n X -> shape: {X.shape}, dtype: {X.dtype}')

# make IY the same shape as FX (and pad the sides)
tmpIy = np.zeros_like(vals[FX])  # create a zero array with the same shape as FX
tmpIy[:, 1:-1, 1:-1] = vals[IY]  # fill the inner part with IY values
vals[IY] = tmpIy  # replace IY with the new array

# reassign the values to the variables
Y = {  
    FX:vals[FX], 
    IY:vals[IY], 
    BR:vals[BR],
    BZ:vals[BZ],
    RQ:vals[RQ],
    ZQ:vals[ZQ],
}

assert Y[FX].shape[0] > 0, f'No samples: {Y[FX].shape}'
N_OR = Y[FX].shape[0]  # number of original samples
print(f'Loaded {N_OR} samples.')

assert X.dtype == DTYPE, f'X dtype mismatch: {X.dtype} != {DTYPE}'
assert all(y.dtype == DTYPE for y in Y.values()), f'Y dtype mismatch: {[y.dtype for y in Y.values()]} != {DTYPE}'

assert X.shape == (N_OR, NIN), f'X shape mismatch: {X.shape} != ({N_OR}, {NIN})'

# check the shapes
print(f'X -> {X.shape}')
print(f'Y -> {[y.shape for y in Y.values()]}')

# print sizes in MB
print(f'X size: {X.nbytes / 1024**2:.2f} MB, Y size: {sum([y.nbytes for y in Y.values()]) / 1024**2:.2f} MB')
print(f'original samples: {tot_samples}')
# assert False

In [ ]:
# lets play with sampling and lcfs
np.set_printoptions(precision=3, suppress=True)
plt.figure(figsize=(16, 12))
rand_idxs = np.random.randint(0, N_OR, 14)
if LOCAL: print(f'Random indices: {rand_idxs}')
for ppli, rand_idx in enumerate(rand_idxs):
    plt.subplot(2, 7, ppli + 1)
    Fx = vals[FX][rand_idx]
    Br, Bz = meqBrBz(Fx)
    rq, zq = filter_zeros(vals[RQ][rand_idx], vals[ZQ][rand_idx])  # RQ and ZQ for the random index
    pq = np.stack((rq, zq), axis=-1)  # stack RQ and ZQ to get the points
    Fx_range = np.max(Fx) - np.min(Fx)
    Fxq = interp_pts(Fx, pq, method='linear')
    # print(f'Fx at lcfs: {np.mean(Fxq):.3f} +/- {np.std(Fxq):.3f}')

    # find all Fx points that are +- 0.01 of the mean of Fxq
    close_LCFS = np.isclose(Fx, np.mean(Fxq), atol=0.08*Fx_range)

    # print(f'Fx LCFS: {np.mean(Fxq):.3f} +/- {np.std(Fxq):.3f}')
    # print(f'Fx range in entire map: min={np.min(Fx):.3f}, max={np.max(Fx):.3f}, range={Fx_range:.3f}')

    #test sampling
    try:
        pts = SAMPLING_FUNC(5000, Fx, rq, zq, σFx=LCFS_FX_SIGMA)
    except AssertionError as e:
        plot_example(
            fx=vals[FX][rand_idx], 
            iy=vals[IY][rand_idx], 
            br=vals[BR][rand_idx], 
            bz=vals[BZ][rand_idx], 
            rq=vals[RQ][rand_idx], 
            zq=vals[ZQ][rand_idx],
            title=f'Error in sampling: {e}'
        )

    plt.contourf(RRD, ZZD, Fx, levels=20, alpha=0.8)
    plt.plot(*filter_zeros(rq, zq), 'red', lw=1, label='LCFS')
    plt.scatter(pts[:, 0], pts[:, 1], c='yellow' if DARK_THEME else 'black', s=1)
    plot_vessel(), plt.axis('equal'), plt.axis('off')
    MIN_NUMBER_NONZERO_IY
plt.tight_layout()
plt.show() if LOCAL else plt.savefig(f'{DS_DIR}/imgs/LCFS_sampling.png')
# assert False, "Stop here"  # stop here for now

In [ ]:
# plot some examples
n_plot = 3 if LOCAL else 15
rand_idxs = np.random.randint(0, N_OR, n_plot)
ri = rand_idxs[0]  # use the first random index for the plot

for i, ri in enumerate(rand_idxs):
    #calc BR and BZ again for the plot
    Br, Bz = meqBrBz(vals[FX][ri])
    vals[BR][ri] = Br
    vals[BZ][ri] = Bz
    plot_example(
        vals[FX][ri][IV], 
        vals[IY][ri][IV], 
        vals[BR][ri][IV], 
        vals[BZ][ri][IV], 
        vals[RQ][ri], 
        vals[ZQ][ri], 
        vals[BM][ri], 
        vals[FF][ri], 
        vals[IA][ri], 
        RRD[IV], 
        ZZD[IV], 
        title=f'original_{i}'
    )
del vals  # free memory

In [ ]:
# test interpolation
idx = np.random.randint(0, N_OR)
pts = SAMPLING_FUNC(2000, Y[FX][idx], Y[RQ][idx], Y[ZQ][idx], σFx=LCFS_FX_SIGMA)

def interp_Fx_Br_Bz_adv(fx, pts, k_finer=7, return_or_grid=False):
    assert fx.ndim == 2, f'fx should be 2D, got {fx.ndim}'
    assert pts.ndim == 2 and pts.shape[1] == 2, f'pts should be 2D with shape (N, 2), got {pts.shape}'
    assert fx.shape == (65, 28), f'fx shape mismatch: {fx.shape} != (65, 28)'
    k = k_finer  # grid multiplier for interpolation
    # create a finer grid for interpolation
    rg0, rg1 = RRD[0, 0], RRD[0, -1]  # radial grid range
    zg0, zg1 = ZZD[0, 0], ZZD[-1, 0]  # vertical grid range
    rf = np.linspace(rg0, rg1, (28-1)*k+1)
    zf = np.linspace(zg0, zg1, (65-1)*k+1)
    rrf, zzf = np.meshgrid(rf, zf)  # radial and vertical grid for the finer grid
    rzf = np.stack((rrf, zzf), axis=-1)  # stack to get the points
    pts_to_interp = np.concatenate((pts, rzf.reshape(-1, 2)), axis=0)  # concatenate the points to interpolate
    int_pts = interp_pts(fx, pts_to_interp) # interpolate at the points
    # unpack
    intFx = int_pts[:pts.shape[0]]  # first part is the original points
    Fxf = int_pts[pts.shape[0]:].reshape(rzf.shape[:-1]) # second part is the finer grid
    #calculate Br and Bz using the interpolated Fx
    Brf, Bzf = meqBrBz(Fxf, rr=rrf, zz=zzf)  # calculate Br, Bz from the flux map
    Brf, Bzf = Brf.reshape(rrf.shape), Bzf.reshape(rrf.shape)  # reshape to the original shape
    # interpolate Br, Bz at the original points + the original grid
    if return_or_grid: int_pts = np.concatenate((pts, np.stack((RRD, ZZD), axis=-1).reshape(-1, 2)), axis=0)  
    else: int_pts = pts
    # intBr = interp_pts(Brf, int_pts, gr=rf, gz=zf)
    # intBz = interp_pts(Bzf, int_pts, gr=rf, gz=zf)
    intBr = interp_pts(Brf, int_pts, gr=rf, gz=zf, method='linear')
    intBz = interp_pts(Bzf, int_pts, gr=rf, gz=zf, method='linear')
    if return_or_grid:
        gBr, gBz = intBr[pts.shape[0]:].reshape(RRD.shape), intBz[pts.shape[0]:].reshape(RRD.shape) 
        intBr, intBz = intBr[:pts.shape[0]], intBz[:pts.shape[0]]  # first part is the original points
        return intFx, intBr, intBz, gBr, gBz
    else: return intFx, intBr, intBz

# intFx = interp_pts(Y[FX][idx], pts)
intIy = interp_pts(Y[IY][idx], pts)
# intBr = interp_pts(Y[BR][idx], pts)
# intBz = interp_pts(Y[BZ][idx], pts)
intFx, intBr, intBz, Y[BR][idx], Y[BZ][idx] = interp_Fx_Br_Bz_adv(Y[FX][idx], pts, return_or_grid=True)

plt.figure(figsize=(15, 8))
ms = 2
plt.subplot(2, 4, 1)
plt.scatter(RRD[IV], ZZD[IV], c=Y[FX][idx][IV], s=ms), plt.title('FX')
plt.plot(*filter_zeros(Y[RQ][idx], Y[ZQ][idx]), 'gray', lw=2)
plot_vessel(), plt.axis('equal'), plt.axis('off'), plt.colorbar()
plt.subplot(2, 4, 5)
plt.scatter(pts[:,0], pts[:,1], c=intFx, s=ms), plt.title('Interpolated FX')
plot_vessel(), plt.axis('equal'), plt.axis('off'), plt.colorbar()
plt.subplot(2, 4, 2)
plt.scatter(RRD[IV], ZZD[IV], c=Y[IY][idx][IV], s=ms), plt.title('IY')
plt.plot(*filter_zeros(Y[RQ][idx], Y[ZQ][idx]), 'gray', lw=2)
plot_vessel(), plt.axis('equal'), plt.axis('off'), plt.colorbar()
plt.subplot(2, 4, 6)
plt.scatter(pts[:,0], pts[:,1], c=intIy, s=ms), plt.title('Interpolated IY')
plot_vessel(), plt.axis('equal'), plt.axis('off'), plt.colorbar()
plt.subplot(2, 4, 3)
plt.scatter(RRD[IV], ZZD[IV], c=Y[BR][idx][IV], s=ms), plt.title('BR')
plt.plot(*filter_zeros(Y[RQ][idx], Y[ZQ][idx]), 'gray', lw=2)
plot_vessel(), plt.axis('equal'), plt.axis('off'), plt.colorbar()
plt.subplot(2, 4, 7)
plt.scatter(pts[:,0], pts[:,1], c=intBr, s=ms), plt.title('Interpolated BR')
plot_vessel(), plt.axis('equal'), plt.axis('off'), plt.colorbar()
plt.subplot(2, 4, 4)
plt.scatter(RRD[IV], ZZD[IV], c=Y[BZ][idx][IV], s=ms), plt.title('BZ')
plt.plot(*filter_zeros(Y[RQ][idx], Y[ZQ][idx]), 'gray', lw=2)
plot_vessel(), plt.axis('equal'), plt.axis('off'), plt.colorbar()
plt.subplot(2, 4, 8)
plt.scatter(pts[:,0], pts[:,1], c=intBz, s=ms), plt.title('Interpolated BZ')
plot_vessel(), plt.axis('equal'), plt.axis('off'), plt.colorbar()
plt.suptitle(f'SHOT {idx} - Interpolation Example')
plt.tight_layout()
plt.show() if LOCAL else plt.savefig(join(DS_DIR, 'imgs', 'interpolation_example.png'))
plt.close()

In [ ]:
# dataset splitting (N_TOP = original dataset size)
NT = int(N_SAMPLES*TRAIN_EVAL_SPLIT)    # training
NE = N_SAMPLES - NT                     # evaluation
print(f"Train -> NT:{NT}")
print(f"Eval  -> NE:{NE}")
orig_idxs = np.random.permutation(N_OR)
orig_idxs_train = orig_idxs[:int(N_OR*TRAIN_EVAL_SPLIT)] # original indices for training
orig_idxs_eval = orig_idxs[int(N_OR*TRAIN_EVAL_SPLIT):] # original indices for evaluation
# splitting the idxs
assert len(orig_idxs_train) > NT, f"Training set is too small, {len(orig_idxs_train)} < {NT}"
idxs_t = np.random.choice(orig_idxs_train, NT, replace=False) 
assert len(orig_idxs_eval) > NE, f"Evaluation set is too small, {len(orig_idxs_eval)} < {NE}"
idxs_e = np.random.choice(orig_idxs_eval, NE, replace=False) 

In [ ]:
import multiprocessing
# create arrays to store the dataset
print(f"Preallocating arrays for the dataset...")
vt = {
    PHYS:   np.zeros((NT, NIN), dtype=DTYPE),
    PTS:    np.zeros((NT, SM, 2), dtype=DTYPE),
    FX:     np.zeros((NT, SM), dtype=DTYPE),
    IY:     np.zeros((NT, SM), dtype=DTYPE),
    BR:     np.zeros((NT, SM), dtype=DTYPE),
    BZ:     np.zeros((NT, SM), dtype=DTYPE),
    SEP:    np.zeros((NT, 2*NLCFS), dtype=DTYPE),
}
ve = {
    PHYS:   np.zeros((NE, NIN), dtype=DTYPE),
    PTS:    np.zeros((NE, SM, 2), dtype=DTYPE),
    FX:     np.zeros((NE, SM), dtype=DTYPE),
    IY:     np.zeros((NE, SM), dtype=DTYPE),
    BR:     np.zeros((NE, SM), dtype=DTYPE),
    BZ:     np.zeros((NE, SM), dtype=DTYPE),
    SEP:    np.zeros((NE, 2*NLCFS), dtype=DTYPE),
    FG:     np.zeros((NE, 4, 65, 28), dtype=DTYPE), # store the full grid maps for evaluation
}

# estimate RAM usage
print(f"Estimated RAM usage: {(sum(arr.nbytes for arr in vt.values()) + sum(arr.nbytes for arr in ve.values())) / (1024**3):.2f} GB")
print("Filling arrays...")

def process_train_iteration(args):
    """
    Processes a single iteration for the training dataset.
    This function is used by both sequential and parallel execution paths.
    """
    i, idx = args
    phys_val = X[idx]
    rq, zq, fx = Y[RQ][idx], Y[ZQ][idx], Y[FX][idx]
    sep_val = np.concatenate([rq, zq], axis=0)
    pts = SAMPLING_FUNC(SM, fx, rq, zq, σFx=LCFS_FX_SIGMA)
    iy_val = interp_pts(Y[IY][idx], pts, method='linear')
    fx_val, br_val, bz_val = interp_Fx_Br_Bz_adv(Y[FX][idx], pts)
    del fx, rq, zq  # clear to save memory
    return i, phys_val, sep_val, pts, iy_val, fx_val, br_val, bz_val

def process_eval_iteration(args):
    """
    Processes a single iteration for the evaluation dataset.
    This function is used by both sequential and parallel execution paths.
    """
    i, idx = args
    phys_val = X[idx]
    rq, zq, fx = Y[RQ][idx], Y[ZQ][idx], Y[FX][idx]
    sep_val = np.concatenate([rq, zq], axis=0)
    pts = SAMPLING_FUNC(SM, fx, rq, zq, σFx=LCFS_FX_SIGMA)
    iy_val = interp_pts(Y[IY][idx], pts)
    fxp, brp, bzp, brg, bzg = interp_Fx_Br_Bz_adv(Y[FX][idx], pts, return_or_grid=True)
    fg_val = (Y[FX][idx], Y[IY][idx], brg, bzg)
    del fx, rq, zq  # clear to save memory
    return i, phys_val, sep_val, pts, iy_val, fxp, brp, bzp, fg_val

## TRAIN
print(f"Processing training data...")
if LOCAL:
    iter_train = tqdm(enumerate(idxs_t), total=NT, desc="Filling training data", unit="sample")
    for i, idx in iter_train:
        _, phys_val, sep_val, pts_val, iy_val, fx_val, br_val, bz_val = process_train_iteration((i, idx))
        vt[PHYS][i], vt[SEP][i], vt[PTS][i], vt[IY][i], vt[FX][i], vt[BR][i], vt[BZ][i] = \
            phys_val, sep_val, pts_val, iy_val, fx_val, br_val, bz_val
else:
    print(f'parallel on {NCORES} cores')
    print_every = 20000
    # Process training data in parallel
    start_time = time()
    with multiprocessing.Pool(processes=NCORES) as pool:
        results_gen = pool.imap_unordered(process_train_iteration, enumerate(idxs_t))
        # Since imap_unordered returns results out of order, we need a separate counter
        for processed_count, result in enumerate(results_gen, 1):
            i, phys_val, sep_val, pts_val, iy_val, fx_val, br_val, bz_val = result
            vt[PHYS][i], vt[SEP][i], vt[PTS][i], vt[IY][i], vt[FX][i], vt[BR][i], vt[BZ][i] = \
                phys_val, sep_val, pts_val, iy_val, fx_val, br_val, bz_val
            
            if processed_count % print_every == 0 or processed_count == NT:
                elapsed = time() - start_time
                eta_seconds = (elapsed / processed_count) * (NT - processed_count) if processed_count > 0 else 0
                print(f"Train -> {100*processed_count/NT:.1f}%, eta: {eta_seconds/60:.1f} min")

# keep only the eval inputs to save memory
for k in [FX,IY,BR,BZ,RQ,ZQ]: Y[k] = Y[k][idxs_e]  
X = X[idxs_e] 

# normalize, NOTE: y3 does not require normalization
print("Getting normalization constants for the dataset...")
μx, Σx = np.mean(vt[PHYS], axis=0), np.std(vt[PHYS], axis=0)  # mean and std of the physical inputs
print(f'μx: {μx.shape}, Σx: {Σx.shape}')
x_mean_std = np.array([μx, Σx])
print(f'x_mean_std: {x_mean_std.shape}')

# save da3330792taset as numpy compressed
print(f"Saving datasets to {TRAIN_DS_PATH}...")
np.savez_compressed(TRAIN_DS_PATH, **vt, x_mean_std=x_mean_std)
print(f"Datasets saved to {TRAIN_DS_PATH}.")
print(f"Train -> PHYS:{vt[PHYS].shape}, PTS:{vt[PTS].shape}, FX:{vt[FX].shape}, IY:{vt[IY].shape}, BR:{vt[BR].shape}, BZ:{vt[BZ].shape}, SEP:{vt[SEP].shape}")
del vt  # free memory

print(f"Processing evaluation data...")
if LOCAL:
        # iter_eval = tqdm(enumerate(idxs_e), total=NE, desc="Filling evaluation data", unit="sample")
    iter_eval = tqdm(enumerate(range(NE)), total=NE, desc="Filling evaluation data", unit="sample")
    for i, idx in iter_eval:
        _, phys_val, sep_val, pts_val, iy_val, fx_val, br_val, bz_val, fg_val = process_eval_iteration((i, idx))
        ve[PHYS][i], ve[SEP][i], ve[PTS][i], ve[IY][i], ve[FX][i], ve[BR][i], ve[BZ][i], ve[FG][i] = \
            phys_val, sep_val, pts_val, iy_val, fx_val, br_val, bz_val, fg_val
else:
    # Process evaluation data in parallel
    start_time = time()
    with multiprocessing.Pool(processes=NCORES) as pool:
        # results_gen = pool.imap_unordered(process_eval_iteration, enumerate(idxs_e))
        results_gen = pool.imap_unordered(process_eval_iteration, enumerate(range(NE)))  
        for processed_count, result in enumerate(results_gen, 1):
            i, phys_val, sep_val, pts_val, iy_val, fx_val, br_val, bz_val, fg_val = result
            ve[PHYS][i], ve[SEP][i], ve[PTS][i], ve[IY][i], ve[FX][i], ve[BR][i], ve[BZ][i], ve[FG][i] = \
                phys_val, sep_val, pts_val, iy_val, fx_val, br_val, bz_val, fg_val

            if processed_count % print_every == 0 or processed_count == NE:
                elapsed = time() - start_time
                eta_seconds = (elapsed / processed_count) * (NE - processed_count) if processed_count > 0 else 0
                print(f"Eval -> {100*processed_count/NE:.1f}%, eta: {eta_seconds/60:.1f} min")

# save dataset as numpy compressed
print(f"Saving datasets to {EVAL_DS_PATH}...")
np.savez_compressed(EVAL_DS_PATH, **ve, x_mean_std=x_mean_std)
print(f"Datasets saved to {EVAL_DS_PATH}.")
print(f"Eval  -> PHYS:{ve[PHYS].shape}, PTS:{ve[PTS].shape}, FX:{ve[FX].shape}, IY:{ve[IY].shape}, BR:{ve[BR].shape}, BZ:{ve[BZ].shape}, SEP:{ve[SEP].shape}, FG:{ve[FG].shape}")

In [ ]:
# save a .mat demo file for testing
print(f"Saving demo file to {DS_DIR}/demo.mat and {DS_DIR}/demo.npz")
rand_idxs = np.random.randint(0, len(ve[PHYS]), 100)  # random indices for the demo
demo = {PHYS:ve[PHYS][rand_idxs], PTS:ve[PTS][rand_idxs], FX:ve[FX][rand_idxs], IY:ve[IY][rand_idxs], BR:ve[BR][rand_idxs], BZ:ve[BZ][rand_idxs], SEP:ve[SEP][rand_idxs]}
try:
    savemat(join(DS_DIR, 'demo.mat'), demo)
    np.savez_compressed(join(DS_DIR, 'demo.npz'), **demo)
except Exception as e:
    print(f"Error saving demo file: {e}")
    raise e
del ve

In [ ]:
# plot some examples
print("Testing the dataset...")
tds, eds = np.load(TRAIN_DS_PATH), np.load(EVAL_DS_PATH)
print(f'train_ds: {tds.keys()}')
print(f'eval_ds: {eds.keys()}')
# plot some examples
n_plot = 3 if LOCAL else 50
for ds, ds_name in zip([tds, eds], ['Train', 'Eval']):
    for i in range(n_plot):
        r, z = ds[PTS][i, :, 0], ds[PTS][i, :, 1]  # get the points
        x = (ds[PHYS][i]-ds['x_mean_std'][0]) / ds['x_mean_std'][1]  # normalize the physical inputs
        plt.figure(figsize=(15, 4))
        plt.subplot(1, 5, 1)
        plt.scatter(r, z, c=ds[FX][i], s=4), plt.title('FX')
        plt.plot(*filter_zeros(ds[SEP][i, :NLCFS], ds[SEP][i, NLCFS:]), 'gray', lw=2)
        plot_vessel(), plt.axis('equal'), plt.axis('off'), plt.colorbar()
        plt.subplot(1, 5, 2)
        plt.scatter(r, z, c=ds[IY][i], s=4), plt.title('IY')
        plt.plot(*filter_zeros(ds[SEP][i, :NLCFS], ds[SEP][i, NLCFS:]), 'gray', lw=2)
        plot_vessel(), plt.axis('equal'), plt.axis('off'), plt.colorbar()
        plt.subplot(1, 5, 3)
        plt.scatter(r, z, c=ds[BR][i], s=4), plt.title('BR')
        plt.plot(*filter_zeros(ds[SEP][i, :NLCFS], ds[SEP][i, NLCFS:]), 'gray', lw=2)
        plot_vessel(), plt.axis('equal'), plt.axis('off'), plt.colorbar()
        plt.subplot(1, 5, 4)
        plt.scatter(r, z, c=ds[BZ][i], s=4), plt.title('BZ')
        plt.plot(*filter_zeros(ds[SEP][i, :NLCFS], ds[SEP][i, NLCFS:]), 'gray', lw=2)
        plot_vessel(), plt.axis('equal'), plt.axis('off'), plt.colorbar()
        plt.subplot(1, 5, 5)
        plt.bar(np.arange(len(x)), x), plt.title('Physical Inputs')
        plt.suptitle(f'SHOT {i} - {ds_name} Example')
        plt.tight_layout()
        plt.show() if LOCAL else plt.savefig(join(DS_DIR, 'imgs', f'{ds_name.lower()}_example_{i}.png'))
        plt.close()

In [ ]:
print('Done! Space used:')
os.system(f'du -h {TRAIN_DS_PATH}')
os.system(f'du -h {EVAL_DS_PATH}')
assert os.path.exists(TRAIN_DS_PATH), f"Dataset not saved: {TRAIN_DS_PATH}"
assert os.path.exists(EVAL_DS_PATH), f"Dataset not saved: {EVAL_DS_PATH}"
print(f"{JOBID} done")
if not LOCAL: sleep(30) # wait for files to update (for cluster)